# MATWM + ICM (Intrinsic Curiosity Module) Intrinsic Rewards

## 目的

非中央集権型MARL（Multi-Agent Reinforcement Learning）において、
**ICM（Intrinsic Curiosity Module）による内発的報酬**を導入し、探索効率を改善する。

- **環境**: PettingZoo `simple_tag_v3`（3 追手 vs 1 逃走者）
- **ベースモデル**: MATWM (Multi-Agent Transformer World Model)
- **好奇心手法**: ICM (Pathak et al. 2017)

---

## ICM (Intrinsic Curiosity Module) とは

### 基本原理

ICMは**自分の行動が環境に与える影響の予測誤差**を新規性として測定する手法。

RNDとの違い：
- **RND**: 観測のみで新規性を判断 → ランダムな要素（ノイズ）にも反応
- **ICM**: 「自分の行動で制御可能な変化」のみを新規性として扱う

### 3つのコンポーネント

1. **Feature Network φ**: 観測を特徴空間に埋め込む
   - 入力: 観測 `s`
   - 出力: 特徴ベクトル `φ(s)`

2. **Inverse Model**: 現在と次の状態から行動を予測
   - 入力: `φ(s_t)`, `φ(s_{t+1})`
   - 出力: 予測行動 `â_t`
   - 目的: 特徴空間が「行動の影響」を捉えるように学習

3. **Forward Model**: 現在の状態と行動から次の状態を予測
   - 入力: `φ(s_t)`, `a_t`
   - 出力: 予測次状態特徴 `φ̂(s_{t+1})`
   - 目的: 状態遷移を学習

### 内発的報酬

```
r_intrinsic = || φ̂(s_{t+1}) - φ(s_{t+1}) ||^2
```

Forward Modelの予測誤差 = 「自分の行動の影響を予測できなかった度合い」

### なぜ機能するか

- **制御可能な遷移**: Forward Model が予測可能 → 低い報酬
- **制御不可能な遷移**: Forward Model が予測困難 → 高い報酬
- **Inverse Model の役割**: 特徴空間が「行動で制御可能な情報」のみを保持するように制約

→ ランダムなノイズは無視され、「自分の行動が重要な影響を持つ状態」を探索

### 利点

- **Noisy TV 問題の回避**: ランダムな要素を無視
- **エージェント中心**: 自分の行動が環境に与える影響を重視
- **Feature Learning**: タスクに関連する特徴を自動学習

---

## なぜ探索が必要か（非中央集権型MARLにおけるモチベーション）

### 1. 部分観測性と信用割当の困難
各エージェントは局所的な観測しか得られず、環境報酬の変動が自分の行動の結果なのか、
他エージェントの行動の結果なのか区別できない。

### 2. 探索の局所化
中央集権型では全体の状態空間を見渡して探索を誘導できるが、
非中央集権型では「全員が同じ局所最適に陥る」相関探索問題が発生する。
ICM は各エージェントに「自分の行動が重要な場所」を探索する動機を与える。

### 3. 報酬の希薄性
追手チームが獲物を捕まえるまで有意な報酬が得られない。
ICM による内発的報酬は、自分の行動の影響に基づく密な報酬シグナルを提供する。

## 1. セットアップ

In [ ]:
# Add project root to Python path for importing modules
import sys
from pathlib import Path

# Get project root (2 levels up from current notebook location)
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root added to path: {project_root}")

In [ ]:
# 必要パッケージのインストール
%pip install pygame
%pip install --no-deps pettingzoo
%pip install numpy gymnasium supersuit
%pip install torchinfo

import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 環境の作成と仕様確認
from pettingzoo.mpe import simple_tag_v3

def make_env(max_cycles=25, seed=None):
    env = simple_tag_v3.parallel_env(
        num_good=1,
        num_adversaries=3,
        num_obstacles=2,
        max_cycles=max_cycles,
        continuous_actions=False,
        render_mode=None,
    )
    if seed is not None:
        env.reset(seed=seed)
    return env

env = make_env(seed=0)
obs, info = env.reset(seed=0)

print('=== Environment Specifications ===')
print('Agents:', env.agents)
print('\nObservation shapes:')
for agent in env.agents:
    print(f'  {agent}: {obs[agent].shape}')
print('\nAction spaces:')
for agent in env.agents:
    print(f'  {agent}: {env.action_space(agent)}')
env.close()

## 2. MATWM 実装の読み込み

In [ ]:
from matwm_implementation import MATWMConfig, pad_observation
from matwm_utils import (
    initialize_matwm_weights, init_weights,
    save_full_checkpoint, load_full_checkpoint,
    plot_training_progress,
    inspect_matwm_architecture,
    print_gpu_info, setup_matwm_training
)
from matwm_agent import MATWMAgent

# 設定
config = MATWMConfig(
    total_steps=50000,   # 論文再現時は50000
    warmup_steps=1000,   # 論文再現時は1000
    log_interval=100,
    save_interval=5000,
    use_gamma_progress=False  # ICM実験ではγ-Progressは使用しない
)

print('=== MATWM Configuration ===')
print(f'  Total steps: {config.total_steps}')
print(f'  Warmup steps: {config.warmup_steps}')
print(f'  Latent dim: {config.latent_dim}x{config.num_classes}')
print(f'  Imagination horizon: {config.imagination_horizon}')
print(f'  Max observation dim: {config.max_obs_dim} (zero-padding)')

In [ ]:
# 観測パディングのテスト
print('=== Observation Padding Test ===')
test_obs_14 = np.random.randn(14)
test_obs_16 = np.random.randn(16)

padded_14 = pad_observation(test_obs_14, config.max_obs_dim)
padded_16 = pad_observation(test_obs_16, config.max_obs_dim)

print(f'14 dim -> {padded_14.shape}, last 2 = {padded_14[-2:]} (should be 0)')
print(f'16 dim -> {padded_16.shape}, unchanged = {np.allclose(test_obs_16, padded_16)}')
print('✓ Zero-padding OK')

In [ ]:
# GPU環境とアーキテクチャ確認
gpu_info = print_gpu_info()
setup_info = setup_matwm_training(config, device)

# 共有World Modelの作成と検証
shared_world_model, shared_wm_optimizer = MATWMAgent.create_shared_world_model(config, device)
dummy_agent = MATWMAgent(config, 'adversary_0', 0, device, shared_world_model=shared_world_model)

print('\n=== Weight Initialization ===')
initialize_matwm_weights(shared_world_model, dummy_agent.actor, dummy_agent.critic)

inspect_matwm_architecture(shared_world_model, dummy_agent.actor, dummy_agent.critic, config, device)

## 3. ICM モジュールの設定

ICM (Intrinsic Curiosity Module) の設定:
- **obs_dim**: 観測次元（16次元にパディング済み）
- **action_dim**: 行動次元（5: no-op, left, right, down, up）
- **hidden_dim**: 隠れ層の次元
- **feature_dim**: Feature Network の出力特徴次元
- **forward_weight**: Forward model loss の重み
- **inverse_weight**: Inverse model loss の重み
- **intrinsic_reward_weight**: ICM 内発的報酬の重み

In [ ]:
from curiosity_rnd_icm import ICMConfig, create_icm_managers
import time

# Experiment configuration
EXPERIMENT_METHOD = 'icm'
TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')

# Output directories with method and timestamp
LOG_DIR = f'../../logs/{EXPERIMENT_METHOD}/{TIMESTAMP}'
RESULTS_DIR = f'../../results/{EXPERIMENT_METHOD}/{TIMESTAMP}'

icm_config = ICMConfig(
    obs_dim=16,              # 観測次元（パディング後）
    action_dim=5,            # 行動次元
    hidden_dim=256,          # 隠れ層の次元
    feature_dim=128,         # Feature Network の出力次元
    forward_weight=0.2,      # Forward model loss の重み
    inverse_weight=0.8,      # Inverse model loss の重み
    intrinsic_reward_weight=1.0,  # ICM内発的報酬の重み
    normalize_observations=True,  # 観測の正規化
    normalize_rewards=True,       # 報酬の正規化
)

print('=== ICM Configuration ===')
print(f'  Observation dim: {icm_config.obs_dim}')
print(f'  Action dim: {icm_config.action_dim}')
print(f'  Hidden dim: {icm_config.hidden_dim}')
print(f'  Feature dim: {icm_config.feature_dim}')
print(f'  Forward weight: {icm_config.forward_weight}')
print(f'  Inverse weight: {icm_config.inverse_weight}')
print(f'  Intrinsic reward weight: {icm_config.intrinsic_reward_weight}')
print(f'  Log dir: {LOG_DIR}')

## 4. 訓練関数の定義

ICM による内発的報酬を統合した訓練ループ:

1. Actor Network が行動を選択（warmup中はランダム）
2. 環境ステップを実行
3. ICM が (s, a, s') から内発的報酬を計算
   - Forward Model: φ(s), a → φ̂(s')
   - 内発的報酬: ||φ̂(s') - φ(s')||²
4. `env_reward + intrinsic_reward` を replay buffer に格納
5. ICM (Feature Net, Forward Model, Inverse Model) を訓練
6. World Model + Actor-Critic を訓練

In [ ]:
def train_matwm_with_icm(config, icm_config, save_dir='results', resume_from=None):
    """
    MATWM + ICM 内発的報酬で訓練。

    Args:
        config: MATWMConfig
        icm_config: ICMConfig
        save_dir: 保存先ディレクトリ
        resume_from: チェックポイントからの再開パス
    """
    # 環境作成
    env = make_env(max_cycles=config.max_cycles, seed=42)
    agent_names = env.agents

    # 共有 World Model
    shared_wm, shared_wm_opt = MATWMAgent.create_shared_world_model(config, device)
    print(f'Shared World Model: {sum(p.numel() for p in shared_wm.parameters())} params')

    # エージェント作成
    agents = {}
    for idx, name in enumerate(agent_names):
        agents[name] = MATWMAgent(config, name, idx, device, shared_world_model=shared_wm)

    # 重み初期化
    if resume_from is None:
        print('\n=== Initializing Weights ===')
        initialize_matwm_weights(shared_wm,
                                 list(agents.values())[0].actor,
                                 list(agents.values())[0].critic)
        for agent in agents.values():
            agent.actor.apply(init_weights)
            agent.critic.apply(init_weights)
        print('✓ Weight initialization complete')

    # ICM マネージャ作成
    icm_managers = create_icm_managers(
        agent_names,
        icm_config,
        device,
    )
    print(f'\n=== ICM Managers Created ===')  
    print(f'  {len(icm_managers)} agents with Feature/Forward/Inverse models')

    # メトリクス
    episode_rewards = {name: [] for name in agent_names}
    episode_curiosity = {name: [] for name in agent_names}  # ICM報酬の推移
    training_metrics = defaultdict(list)
    icm_metrics = defaultdict(list)
    start_step = 0

    # チェックポイントからの再開
    if resume_from is not None and os.path.exists(resume_from):
        print(f'\n=== Resuming from: {resume_from} ===')
        episode_rewards, training_metrics, start_step = load_full_checkpoint(
            agents, shared_wm, shared_wm_opt, resume_from, device
        )
        print(f'✓ Resumed from step {start_step}')

    # 保存ディレクトリ
    os.makedirs(save_dir, exist_ok=True)
    timestamp = time.strftime('%Y_%m_%d_%H_%M_%S')
    run_dir = os.path.join(save_dir, f'matwm_icm_{timestamp}')
    os.makedirs(run_dir, exist_ok=True)

    print(f'\n=== Starting MATWM + ICM Training ===')
    print(f'Save directory: {run_dir}')
    print(f'Total steps: {config.total_steps}')
    print(f'Warmup steps: {config.warmup_steps}')
    print(f'ICM weight: {icm_config.intrinsic_reward_weight}\n')

    # 訓練ループ
    global_step = start_step
    episode_count = 0
    min_data = config.wm_batch_length + 10
    pbar = tqdm(total=config.total_steps, initial=start_step, desc='Training')

    while global_step < config.total_steps:
        obs, info = env.reset()
        ep_reward = {name: 0.0 for name in agent_names}
        ep_intrinsic = {name: 0.0 for name in agent_names}

        # エピソード開始: ICMマネージャをリセット
        for name in agent_names:
            icm_managers[name].reset_episode(episode_count)

        for step in range(config.max_cycles):
            # 行動選択
            actions = {}
            for name, agent in agents.items():
                if global_step < config.warmup_steps:
                    actions[name] = env.action_space(name).sample()
                else:
                    actions[name] = agent.select_action(obs[name])

            # 環境ステップ
            next_obs, rewards, terms, truncs, infos = env.step(actions)
            done = {name: terms[name] or truncs[name] for name in agent_names}

            # ICM 内発的報酬の計算
            for name, agent in agents.items():
                other_acts = {k: v for k, v in actions.items() if k != name}
                env_r = rewards[name]

                # ICM 内発的報酬（obs, action, next_obs を使用）
                intrinsic_r = 0.0
                if global_step >= min_data:
                    obs_padded = pad_observation(obs[name], config.max_obs_dim)
                    next_obs_padded = pad_observation(next_obs[name], config.max_obs_dim)
                    intrinsic_r = icm_managers[name].compute_intrinsic_reward(
                        obs_padded, actions[name], next_obs_padded
                    )

                total_r = env_r + intrinsic_r

                # Replay buffer に格納
                agent.store_experience(
                    obs[name], actions[name], total_r,
                    next_obs[name], done[name], other_acts,
                )
                ep_reward[name] += env_r
                ep_intrinsic[name] += intrinsic_r

            obs = next_obs
            global_step += 1
            pbar.update(1)

            # World Model 訓練
            if global_step >= config.warmup_steps:
                wm_metrics = MATWMAgent.train_world_model_shared(
                    agents, config, device, shared_wm_opt
                )
                if wm_metrics:
                    for k, v in wm_metrics.items():
                        training_metrics[f'shared_{k}'].append(v)

                # Actor-Critic 訓練
                for name, agent in agents.items():
                    ac_metrics = agent.train_agent()
                    for k, v in ac_metrics.items():
                        training_metrics[f'{name}_{k}'].append(v)

                # ICM 訓練（定期的に）
                if global_step % 10 == 0:  # 10ステップごと
                    for name, agent in agents.items():
                        # replay buffer からサンプル
                        if len(agent.replay_buffer) >= config.wm_batch_size:
                            batch = agent.replay_buffer.sample(config.wm_batch_size)
                            obs_batch = np.array([pad_observation(exp[0], config.max_obs_dim) for exp in batch])
                            action_batch = np.array([exp[1] for exp in batch])
                            next_obs_batch = np.array([pad_observation(exp[3], config.max_obs_dim) for exp in batch])
                            
                            icm_loss_dict = icm_managers[name].train(obs_batch, action_batch, next_obs_batch)
                            for k, v in icm_loss_dict.items():
                                icm_metrics[f'{name}_{k}'].append(v)

            # ログ
            if global_step % config.log_interval == 0 and global_step >= config.warmup_steps:
                log_str = f'Step {global_step}: '
                for name in agent_names:
                    if episode_rewards[name]:
                        log_str += f'{name}={np.mean(episode_rewards[name][-10:]):.2f} '
                pbar.set_description(log_str)

            # チェックポイント保存
            if global_step % config.save_interval == 0 and global_step >= config.warmup_steps:
                ckpt_dir = os.path.join(run_dir, f'checkpoint_{global_step}')
                os.makedirs(ckpt_dir, exist_ok=True)
                for name, agent in agents.items():
                    agent.save(os.path.join(ckpt_dir, f'{name}.pt'))
                save_full_checkpoint(
                    agents, shared_wm, shared_wm_opt,
                    episode_rewards, training_metrics, global_step,
                    os.path.join(ckpt_dir, 'full_checkpoint.pt')
                )
                print(f'\n✓ Checkpoint saved at step {global_step}')

            if all(done.values()):
                break

        # エピソード終了
        for name in agent_names:
            icm_managers[name].end_episode()
            episode_rewards[name].append(ep_reward[name])
            episode_curiosity[name].append(ep_intrinsic[name])

        episode_count += 1

    pbar.close()
    env.close()

    # 最終チェックポイント
    final_dir = os.path.join(run_dir, 'final')
    os.makedirs(final_dir, exist_ok=True)
    for name, agent in agents.items():
        agent.save(os.path.join(final_dir, f'{name}.pt'))
    save_full_checkpoint(
        agents, shared_wm, shared_wm_opt,
        episode_rewards, training_metrics, global_step,
        os.path.join(final_dir, 'full_checkpoint.pt')
    )

    print(f'\n=== Training Complete ===')
    print(f'Total episodes: {episode_count}')
    print(f'Final checkpoint: {final_dir}')
    for name in agent_names:
        if episode_rewards[name]:
            r = episode_rewards[name][-100:] if len(episode_rewards[name]) >= 100 else episode_rewards[name]
            print(f'  {name}: mean reward = {np.mean(r):.2f}')

    # ICM統計
    print('\n=== ICM Statistics ===')
    for name in agent_names:
        if episode_curiosity[name]:
            c = episode_curiosity[name]
            print(f'  {name}: mean intrinsic reward = {np.mean(c):.2f}')

    return agents, episode_rewards, training_metrics, episode_curiosity, icm_metrics

print('Training function defined. ✓')

## 5. 訓練の実行

**注意**: 完全な訓練には時間がかかります（GPUで数時間）。
短時間テストの場合は `config.total_steps` を小さくしてください。

In [ ]:
# 訓練実行
print('=' * 70)
print('MATWM + ICM Training')
print('=' * 70)

agents, episode_rewards, training_metrics, episode_curiosity, icm_metrics = \
    train_matwm_with_icm(config, icm_config, save_dir=RESULTS_DIR)

# チェックポイントからの再開（コメントアウト解除して使用）
# checkpoint_path = '../../results/icm/YYYYMMDD_HHMMSS/checkpoint_25000/full_checkpoint.pt'
# agents, episode_rewards, training_metrics, episode_curiosity, icm_metrics = \
#     train_matwm_with_icm(config, icm_config, save_dir=RESULTS_DIR, resume_from=checkpoint_path)

## 6. エージェントの評価

In [ ]:
def evaluate_agents(agents, num_episodes=20):
    """訓練済みエージェントの性能を評価"""
    env = make_env(max_cycles=config.max_cycles)
    agent_names = list(agents.keys())
    eval_rewards = {name: [] for name in agent_names}

    for ep in range(num_episodes):
        obs, _ = env.reset()
        ep_reward = {name: 0.0 for name in agent_names}

        for step in range(config.max_cycles):
            actions = {name: agent.select_action(obs[name], deterministic=True)
                       for name, agent in agents.items()}
            next_obs, rewards, terms, truncs, _ = env.step(actions)

            for name in agent_names:
                ep_reward[name] += rewards[name]

            obs = next_obs
            if all(terms[n] or truncs[n] for n in agent_names):
                break

        for name in agent_names:
            eval_rewards[name].append(ep_reward[name])

        print(f'Episode {ep+1}/{num_episodes}: ' +
              ' '.join(f'{n}={ep_reward[n]:.2f}' for n in agent_names))

    env.close()

    print('\n=== Evaluation Results ===')
    for name in agent_names:
        print(f'  {name}: Mean={np.mean(eval_rewards[name]):.2f}, Std={np.std(eval_rewards[name]):.2f}')

    return eval_rewards

print('Evaluating trained agents...')
eval_rewards = evaluate_agents(agents, num_episodes=20)

## 7. 訓練結果の可視化

In [ ]:
# 詳細可視化
print('=' * 70)
print('TRAINING VISUALIZATION')
print('=' * 70)
plot_training_progress(episode_rewards, training_metrics, save_path='results/training_curves_icm.png')

In [ ]:
# 学習曲線 + ICM報酬の推移
agent_names = list(episode_rewards.keys())

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# (0,0) Episode rewards
ax = axes[0, 0]
for name in agent_names:
    r = episode_rewards[name]
    if len(r) > 0:
        w = min(10, len(r))
        if len(r) >= w:
            ma = np.convolve(r, np.ones(w)/w, mode='valid')
            ax.plot(ma, label=name)
ax.set_title('Episode Rewards (Moving Average)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# (0,1) Shared WM Total Loss
ax = axes[0, 1]
key = 'shared_wm_total_loss'
if key in training_metrics and training_metrics[key]:
    ax.plot(training_metrics[key], alpha=0.7)
ax.set_title('Shared World Model Loss')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)

# (1,0) Actor Loss
ax = axes[1, 0]
for name in agent_names:
    key = f'{name}_actor_loss'
    if key in training_metrics and training_metrics[key]:
        ax.plot(training_metrics[key], label=name, alpha=0.7)
ax.set_title('Actor Loss')
ax.set_xlabel('Training Step')
ax.legend()
ax.grid(True, alpha=0.3)

# (1,1) ICM Forward & Inverse Loss
ax = axes[1, 1]
for name in agent_names:
    key_fwd = f'{name}_forward_loss'
    key_inv = f'{name}_inverse_loss'
    if key_fwd in icm_metrics and icm_metrics[key_fwd]:
        ax.plot(icm_metrics[key_fwd], label=f'{name} (fwd)', alpha=0.7, linestyle='--')
    if key_inv in icm_metrics and icm_metrics[key_inv]:
        ax.plot(icm_metrics[key_inv], label=f'{name} (inv)', alpha=0.7, linestyle='-')
ax.set_title('ICM Forward & Inverse Loss')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# (2,0) ICM Intrinsic Rewards per Episode
ax = axes[2, 0]
for name in agent_names:
    c = episode_curiosity.get(name, [])
    if c:
        ax.plot(c, label=name, alpha=0.7)
ax.set_title('ICM Intrinsic Reward per Episode')
ax.set_xlabel('Episode')
ax.set_ylabel('Intrinsic Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# (2,1) Total Reward (Env + ICM) per Episode
ax = axes[2, 1]
for name in agent_names:
    env_r = episode_rewards.get(name, [])
    icm_r = episode_curiosity.get(name, [])
    if env_r and icm_r:
        total_r = [e + i for e, i in zip(env_r, icm_r)]
        ax.plot(total_r, label=name, alpha=0.7)
ax.set_title('Total Reward (Env + ICM) per Episode')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves_icm_detailed.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/training_curves_icm_detailed.png')

## 8. まとめと考察

### 実装した内容

本 Notebook では、MATWM に **ICM (Intrinsic Curiosity Module) による内発的報酬** を統合した。

#### 主要コンポーネント

1. **World Model**（MATWM 論文通り）
   - Encoder/Decoder (Categorical VAE)
   - Dynamics Model (Transformer)
   - Reward/Continuation Predictor
   - Teammate Predictor

2. **ICM (Intrinsic Curiosity Module)**（本研究の実装）
   - **Feature Network φ**: 観測を特徴空間に埋め込む
   - **Inverse Model**: (φ(s), φ(s')) → a （特徴学習の制約）
   - **Forward Model**: (φ(s), a) → φ̂(s') （遷移予測）
   - **内発的報酬**: 予測誤差 = 自分の行動の影響の新規性

3. **Actor-Critic**（MATWM 論文通り）
   - Actor: 想像軌道上のポリシー学習
   - Critic: 価値関数の推定

---

### ICM の利点と限界

#### 利点

1. **Noisy TV 問題の回避**: ランダムな環境要素を無視
   - Inverse Model が「行動で制御可能な情報」のみを特徴として学習させる
   - ランダムノイズは行動から予測できないため、特徴空間から除外される

2. **エージェント中心の探索**: 「自分の行動が重要な場所」を優先的に探索
   - Forward Model の予測誤差 = 「自分の行動の影響を理解していない状態」
   - より能動的な探索が可能

3. **特徴学習**: タスクに関連する表現を自動獲得
   - 生の観測ではなく、学習された特徴空間で予測
   - 高次元観測（画像など）でも効果的

#### 限界

1. **確率的遷移**: 行動で制御できるが確率的な遷移には弱い
   - Forward Model が予測できない → 常に高報酬
   - ただし RND よりは影響を受けにくい

2. **マルチエージェント環境**:
   - 他エージェントの行動による状態変化を「自分の行動の影響」と誤認する可能性
   - Social Curiosity（TeammatePredictor）の方が適している可能性

3. **計算コスト**: RND より複雑（3つのネットワーク）

---

### RND vs ICM vs Social Curiosity

| 手法 | 対象 | ノイズ耐性 | 多エージェント特化 | 計算コスト |
|------|------|------------|-------------------|------------|
| **RND** | 観測の新規性 | ❌ 低い | ❌ 汎用 | 低 |
| **ICM** | 行動の影響の新規性 | ✅ 高い | ❌ 汎用 | 中 |
| **Social Curiosity** | 他エージェント行動の予測誤差 | ✅ 高い | ✅ 多エージェント固有 | 中 |

---

### 期待される実験結果

- **RND**: 環境の未探索領域を発見
- **ICM**: RND + 行動の影響が大きい領域を優先的に探索
- **Social Curiosity**: ICM + 協調行動パターンの発見

ICM は RND より効果的な探索が期待されるが、
多エージェント環境では Social Curiosity がさらに優位である可能性が高い。

---

### 参考文献

- Pathak et al. (2017). "Curiosity-driven Exploration by Self-supervised Prediction." ICML 2017.
- Burda et al. (2018). "Exploration by Random Network Distillation." ICLR 2019.
- Badia et al. (2020). "Never Give Up: Learning Directed Exploration Strategies." ICLR 2020.